[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_02/MFM_ch2_crypto_momentum/MFM_ch2_crypto_momentum.ipynb)

# MFM_ch2_crypto_momentum

Case study, Liu and Tsyvinski (2021), Risks and returns of cryptocurrency (Review of Financial Studies 34(6), 2689-2727): time-series momentum in weekly Bitcoin returns. Simple weekly returns from Sunday close to Sunday close (UTC); predictive regressions of the return h = 1 to 4 weeks ahead on this week's return, with Newey-West (NW) standard errors, in the paper's sample (January 2011 to May 2018), from 2013, and after the sample (June 2018 to September 2026); quintile sorts on this week's return; the one-week slope over rolling 104-week windows. Ethereum from 2016 for comparison. The paper's numbers (NBER working paper w24877) are shown for reference.

*Modelling Financial Markets (MFM) - Daniel Traian Pele*

In [1]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## Style

In [2]:
# Chart style: transparent background, legend below the plot
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'savefig.facecolor': 'none',
                     'savefig.transparent': True, 'axes.grid': False, 'font.family': 'sans-serif',
                     'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'], 'font.size': 9,
                     'axes.labelsize': 10, 'axes.titlesize': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.6, 'lines.linewidth': 1.1,
                     'legend.facecolor': 'none', 'legend.framealpha': 0, 'legend.fontsize': 8})
MainBlue, IDAred, Amber, Navy, BandBlue = '#1A3A6E', '#CD0000', '#B5853F', '#1F2A44', '#C5D2E8'
CHART_DIR = '.'

## Data

Daily closing prices of Bitcoin and Ethereum from the course data (EODHD), read from the repository.

In [3]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'
SYMBOL = {'btc': 'BTC-USD.CC', 'eth': 'ETH-USD.CC'}


def read_market(symbol):
    """Daily price table of one series (local copy of the course data, otherwise the GitHub repository)."""
    path = os.path.join(MARKET_DIR, f'{symbol}.csv')
    src = path if MARKET_DIR and os.path.exists(path) else REPO_RAW + f'{symbol}.csv'
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def load_close(name, start, end=END):
    """Daily closing price of a cryptocurrency (7 days a week, as in the chapter)."""
    s = read_market(SYMBOL[name])['close'].loc[start:end]
    return s[s > 0].dropna().rename(name)

## Settings

Periods, horizons, rolling window and the numbers of the paper (working paper w24877: Table 14 Panel B, Table 15).

In [4]:
START = {'btc': '2011-01-01', 'eth': '2016-01-01'}
PERIODS = {'paper': ('2011-01-01', '2018-05-31'),           # sample of the paper (NBER w24877)
           'since2013': ('2013-01-01', '2018-05-31'),       # the paper's Table 16
           'after': ('2018-06-01', END)}
HORIZONS = (1, 2, 3, 4)
ROLL = 104
PAPER = dict(beta=[0.19, 0.22, 0.21, 0.09], t_ols=[3.73, 4.52, 4.26, 1.72], sd_pct=16.64, per_sd_pct=3.16,
             top_pct=11.22, top_sharpe=0.45, bottom_pct=2.60, bottom_sharpe=0.19)

## Analysis

Ordinary least squares (OLS) with Newey-West standard errors: Bartlett weights, L = floor(4 (T/100)^(2/9)) lags.

In [5]:
def nw_lag(T):
    return int(np.floor(4 * (T / 100) ** (2 / 9)))


def ols_hac(y, x, L=None):
    """OLS y = a + b x; slope, OLS t, Newey-West t (Bartlett weights, no small-sample correction), R^2."""
    y, x = np.asarray(y, float), np.asarray(x, float)
    T = len(y)
    L = nw_lag(T) if L is None else L
    X = np.column_stack([np.ones(T), x])
    XtXi = np.linalg.inv(X.T @ X)
    b = XtXi @ X.T @ y
    e = y - X @ b
    s2 = e @ e / (T - 2)
    u = X * e[:, None]
    S = u.T @ u
    for l in range(1, L + 1):
        G = u[l:].T @ u[:-l]
        S += (1 - l / (L + 1)) * (G + G.T)
    V = XtXi @ S @ XtXi
    se_hac = np.sqrt(V[1, 1])
    r2 = 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))
    return dict(b=float(b[1]), t_ols=float(b[1] / np.sqrt(s2 * XtXi[1, 1])), t_hac=float(b[1] / se_hac),
                se_hac=float(se_hac), r2=float(r2), T=T, L=L)


def weekly_returns(name):
    """Simple weekly returns, Sunday close to Sunday close (UTC daily closes); only complete weeks."""
    p = load_close(name, start=START[name])
    w = p.resample('W-SUN').last()
    w = w[w.index <= p.index[-1]]
    return w.pct_change().dropna().rename(name)


def period_stats(R, a, b):
    r = R.loc[a:b]
    out = dict(start=str(r.index[0].date()), end=str(r.index[-1].date()), weeks=len(r), sd=float(r.std()), reg={})
    for h in HORIZONS:
        y = r.shift(-h).dropna()
        out['reg'][h] = ols_hac(y.values, r.loc[y.index].values)
    x, y = r.iloc[:-1], r.shift(-1).iloc[:-1]
    q = pd.qcut(x, 5, labels=False)
    g = y.groupby(q.values).agg(['mean', 'std', 'count'])
    top, bot = y[q.values == 4], y[q.values == 0]
    out['quintiles'] = dict(mean=g['mean'].tolist(), sharpe=(g['mean'] / g['std']).tolist(), n=g['count'].tolist(),
                            spread=float(top.mean() - bot.mean()),
                            spread_t=float((top.mean() - bot.mean()) / np.sqrt(top.var() / len(top) + bot.var() / len(bot))))
    return out


def rolling_beta(R, window=ROLL):
    rows = []
    x, y = R.iloc[:-1].values, R.shift(-1).iloc[:-1].values
    for i in range(window, len(x) + 1):
        f = ols_hac(y[i - window:i], x[i - window:i])
        rows.append((R.index[i], f['b'], f['se_hac']))     # date of the last R_{t+1} in the window
    return pd.DataFrame(rows, columns=['date', 'beta', 'se_hac']).set_index('date')

## Chart

In [6]:
def figure(res, roll):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.4, 3.2), gridspec_kw={'width_ratios': [1, 1.45]})
    P, A = res['btc']['paper'], res['btc']['after']
    for k, (d, col, lab, dx) in enumerate([(P, MainBlue, 'Jan 2011 - May 2018 (paper\'s sample)', -0.13),
                                           (A, IDAred, 'Jun 2018 - Sep 2026 (after the sample)', 0.13)]):
        b = np.array([d['reg'][h]['b'] for h in HORIZONS])
        se = np.array([d['reg'][h]['se_hac'] for h in HORIZONS])
        a1.errorbar(np.array(HORIZONS) + dx, b, yerr=1.96 * se, fmt='o', color=col, ms=4, capsize=2.5, lw=1, label=lab)
    a1.plot(HORIZONS, PAPER['beta'], 'D', color=Amber, ms=4.5, label='Liu & Tsyvinski (working paper)')
    a1.axhline(0, color=Navy, lw=0.6)
    a1.set_xticks(HORIZONS, [f'{h}' for h in HORIZONS])
    a1.set_xlabel('Horizon h (weeks)')
    a1.set_ylabel('Slope b_h')
    a1.set_title('Slope of R(t+h) on R(t), 95% NW intervals', fontsize=8.5, loc='left')
    a2.fill_between(roll.index, roll['beta'] - 1.96 * roll['se_hac'], roll['beta'] + 1.96 * roll['se_hac'],
                    color=BandBlue, lw=0, label='95% NW band')
    a2.plot(roll.index, roll['beta'], color=MainBlue, lw=1.1, label='Rolling b_1 (104 weeks)')
    a2.axhline(0, color=Navy, lw=0.6)
    a2.axvline(pd.Timestamp('2018-05-31'), color=IDAred, ls='--', lw=0.9, label="End of paper's sample")
    a2.set_ylabel('Slope b_1')
    a2.set_title('Bitcoin: one-week slope b_1, rolling two-year windows', fontsize=8.5, loc='left')
    plt.tight_layout()
    h1, l1 = a1.get_legend_handles_labels()
    h2, l2 = a2.get_legend_handles_labels()
    fig.legend(h1 + h2, l1 + l2, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=3, frameon=False)
    os.makedirs(CHART_DIR, exist_ok=True)
    for ext in ('pdf', 'png'):
        plt.savefig(os.path.join(CHART_DIR, f'ch2_case_crypto_tsmom.{ext}'), bbox_inches='tight', transparent=True,
                    **({'dpi': 180} if ext == 'png' else {}))
    plt.close()
    print('   saved ch2_case_crypto_tsmom')

## Run

In [7]:
def main():
    R = {k: weekly_returns(k) for k in START}
    res = {'paper_numbers': PAPER, 'roll_window': ROLL}
    for k in R:
        res[k] = {p: period_stats(R[k], a, b) for p, (a, b) in PERIODS.items() if not (k == 'eth' and p == 'since2013')}
    roll = rolling_beta(R['btc'])
    after = roll.loc['2018-06-01':]
    res['btc']['rolling'] = dict(share_sig=float(np.mean(np.abs(roll['beta'] / roll['se_hac']) > 1.96)),
                                 share_sig_after=float(np.mean(np.abs(after['beta'] / after['se_hac']) > 1.96)),
                                 max_beta=float(roll['beta'].max()), max_date=str(roll['beta'].idxmax().date()),
                                 last_beta=float(roll['beta'].iloc[-1]), last_date=str(roll.index[-1].date()),
                                 n=len(roll))
    json.dump(res, open('ch2_crypto_tsmom.json', 'w'), indent=1)
    pd.concat(R, axis=1).to_csv('ch2_crypto_tsmom_weekly.csv', float_format='%.8g')
    roll.to_csv('ch2_crypto_tsmom_rolling.csv', float_format='%.8g')
    figure(res, roll)
    for k in R:
        for p, d in res[k].items():
            if p == 'rolling':
                continue
            print(k, p, d['start'], d['end'], d['weeks'], 'sd %.4f' % d['sd'],
                  ' '.join('h%d b=%.3f t=%.2f tNW=%.2f' % (h, v['b'], v['t_ols'], v['t_hac']) for h, v in d['reg'].items()),
                  'Q', np.round(100 * np.array(d['quintiles']['mean']), 2), 'spread t %.2f' % d['quintiles']['spread_t'])
    print(res['btc']['rolling'])


main()

   saved ch2_case_crypto_tsmom
btc paper 2011-01-09 2018-05-27 386 sd 0.1775 h1 b=0.168 t=3.33 tNW=2.75 h2 b=0.236 t=4.75 tNW=1.98 h3 b=0.168 t=3.33 tNW=2.61 h4 b=0.115 t=2.27 tNW=1.64 Q [ 2.65 -0.63  2.13  5.54  9.93] spread t 2.18
btc since2013 2013-01-06 2018-05-27 282 sd 0.1380 h1 b=0.121 t=2.02 tNW=1.06 h2 b=0.219 t=3.73 tNW=2.52 h3 b=0.126 t=2.11 tNW=2.08 h4 b=0.036 t=0.59 tNW=0.39 Q [2.76 0.17 1.82 4.18 6.84] spread t 1.28
btc after 2018-06-03 2026-09-13 433 sd 0.0831 h1 b=0.115 t=2.41 tNW=2.61 h2 b=0.015 t=0.30 tNW=0.29 h3 b=-0.026 t=-0.54 tNW=-0.58 h4 b=0.072 t=1.48 tNW=1.48 Q [-0.26  1.05  1.27 -1.03  3.36] spread t 2.59
eth paper 2016-01-10 2018-05-27 125 sd 0.2294 h1 b=0.114 t=1.26 tNW=1.65 h2 b=0.119 t=1.32 tNW=1.52 h3 b=0.311 t=3.66 tNW=2.51 h4 b=-0.050 t=-0.56 tNW=-0.62 Q [ 5.89  4.85  8.46  3.52 14.58] spread t 1.17
eth after 2018-06-03 2026-09-13 433 sd 0.1131 h1 b=0.107 t=2.23 tNW=2.45 h2 b=-0.007 t=-0.15 tNW=-0.14 h3 b=-0.017 t=-0.35 tNW=-0.36 h4 b=0.060 t=1.23 tNW=0

![ch2_case_crypto_tsmom](./ch2_case_crypto_tsmom.png)

Output: `ch2_case_crypto_tsmom.pdf`, `ch2_crypto_tsmom.json`, `ch2_crypto_tsmom_weekly.csv`, `ch2_crypto_tsmom_rolling.csv`